# Evaluate AI agents of Foundry Agent Services

## Objective

This sample demonstrates how to:
1. Create a **Foundry prompt agent** (Projects v2) with function tools
2. Run the agent via `openai_client.responses.create()` with function-call dispatch
3. Run a **cloud evaluation** using `openai_client.evals` (the OpenAI Evals API integrated in Foundry)

Evaluators used:
- **Relevance** (AI-assisted) — measures response relevance to the query
- **Violence** (safety) — content safety evaluator
- **BLEU / F1 / METEOR** (NLP) — textual similarity against ground truth

### Before you begin

* An Azure AI Foundry project with a deployed GPT model (e.g. `gpt-4o-mini`)
* **Azure AI User** role on the Foundry project
* Authenticate via `az login` before running

### Prerequisites
```bash
pip install azure-ai-projects>=2.0.0 azure-identity azure-ai-evaluation
```

### Setup Azure credentials and project 
1. use az cli to login to the tenant with your credential

<!-- initializing Project Client -->

In [42]:
from dotenv import load_dotenv

# load environment variables from .env file
load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print("please login first")

Environment and authentication OK


In [43]:
import os
import json
from typing import Any, Dict
import azure.ai.projects as projectslib
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    FunctionTool,
)
from openai.types.responses.response_input_param import (
    FunctionCallOutput,
    ResponseInputParam,
)
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
    SourceFileID,
)

# Import your custom functions to be used as Tools for the Agent
from utils.user_functions import (
    fetch_current_datetime,
    fetch_weather,
    send_email,
    opening_hours,
)

# Initialize project client with proper authentication
project_client = AIProjectClient(
    credential=credential,  # Use the credential from earlier setup
    endpoint=settings.project_endpoint
)
# Get the OpenAI client for evaluation and agent API
openai_client = project_client.get_openai_client()

print("project_client api version:", project_client._config.api_version)
print(f"azure-ai-projects version: {projectslib.__version__}")

project_client api version: v1
azure-ai-projects version: 2.1.0


In [44]:
AGENT_NAME = "seattle-tourist-agent"
AGENT_INSTRUCTIONS = """You are a helpful tourist assistant"""

# In v2, function tools are declared by JSON Schema. The model emits
# `function_call` items in the response; we execute them and feed the
# result back via `FunctionCallOutput`.
tools = [
    FunctionTool(
        name="fetch_current_datetime",
        description="Get the current time as a JSON string, optionally formatted.",
        parameters={
            "type": "object",
            "properties": {
                "format": {"type": ["string", "null"], "description": "The format in which to return the current time. Defaults to None."},
            },
            "required": [],
        },
    ),
    FunctionTool(
        name="fetch_weather",
        description="Fetches the weather information for the specified location.",
        parameters={
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The location to fetch weather for."},
            },
            "required": ["location"],
        },
    ),
    FunctionTool(
        name="send_email",
        description="Sends an email with the specified subject and body to the recipient.",
        parameters={
            "type": "object",
            "properties": {
                "recipient": {"type": "string", "description": "Email address of the recipient."},
                "subject": {"type": "string", "description": "Subject of the email."},
                "body": {"type": "string", "description": "Body content of the email."},
            },
            "required": ["recipient", "subject", "body"],
        },
    ),
    FunctionTool(
        name="opening_hours",
        description="Fetches the opening hours of a tourist destination in Seattle.",
        parameters={
            "type": "object",
            "properties": {
                "tourist_destination": {"type": "string", "description": "The tourist destination to fetch opening hours for."},
            },
            "required": ["tourist_destination"],
        },
    ),
]

# Map function name -> Python callable for the chat loop dispatch
function_dispatch: Dict[str, Any] = {
    "fetch_current_datetime": fetch_current_datetime,
    "fetch_weather": fetch_weather,
    "send_email": send_email,
    "opening_hours": opening_hours,
}

print(f"Prepared {len(tools)} function tools for agent")

Prepared 4 function tools for agent


### Create an AI agent (Azure AI Agent Service)

In [45]:
agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=settings.model_deployment_name,
        instructions=AGENT_INSTRUCTIONS,
        tools=tools,
    ),
    description="Seattle tourist agent with function tools for eval.",
)
print(f"agent > {agent.name} (id: {agent.id}, version: {agent.version})")

agent > seattle-tourist-agent (id: seattle-tourist-agent:1, version: 1)


## Conversation with Agent
In Projects v2, multi-turn state is held by an OpenAI **conversation**.
Use `openai_client.responses.create()` with an `agent_reference` to invoke the agent.

### Create Conversation and Helper - 1

In [46]:
conversation = openai_client.conversations.create()
print(f"Created conversation, ID: {conversation.id}")

AGENT_REFERENCE = {"name": agent.name, "type": "agent_reference"}


def run_agent_once(user_prompt: str, conversation_id: str | None = None):
    """Send user_prompt to the agent and resolve function calls in a loop."""
    kwargs: Dict[str, Any] = {
        "input": user_prompt,
        "extra_body": {"agent_reference": AGENT_REFERENCE},
        "tool_choice": "auto",
    }
    if conversation_id:
        kwargs["conversation"] = conversation_id

    response = openai_client.responses.create(**kwargs)

    while True:
        pending = [item for item in response.output if item.type == "function_call"]
        if not pending:
            return response

        input_list: ResponseInputParam = []
        for item in pending:
            fn = function_dispatch.get(item.name)
            if fn is None:
                output = json.dumps({"error": f"unknown function {item.name}"})
            else:
                args = json.loads(item.arguments) if item.arguments else {}
                print(f"  {item.name}({args})")
                output = fn(**args)
            input_list.append(FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=output,
            ))

        response = openai_client.responses.create(
            input=input_list,
            previous_response_id=response.id,
            extra_body={"agent_reference": AGENT_REFERENCE},
        )
    return response

Created conversation, ID: conv_1d85c4b3bca56a0e00M3f0vndi3GtRLj7JbNWt2v4zA9QubA2D


### Send Message and Execute - 2

In [47]:
MESSAGE = "Can you email me weather info for Seattle ? My email is user@microsoft.com."

response = run_agent_once(MESSAGE, conversation_id=conversation.id)
print(f"Response ID: {response.id}")
print(f"Agent > {response.output_text}")

  fetch_weather({'location': 'Seattle'})
  send_email({'recipient': 'user@microsoft.com', 'subject': 'Weather Information for Seattle', 'body': 'The current weather in Seattle is rainy with a temperature of 14°C.'})
Sending email to user@microsoft.com...
Subject: Weather Information for Seattle
Body:
The current weather in Seattle is rainy with a temperature of 14°C.
Response ID: resp_1d85c4b3bca56a0e0069f912c136148190954cfd2d623c117c
Agent > I have sent you an email with the current weather information for Seattle. It is rainy and 14°C there. If you need any more information, feel free to ask!


# Evaluate

### Evaluation in the cloud

Reference:
* https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/cloud-evaluation?tabs=python
* https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/cloud-evaluation#prepare-input-data
* https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent
* https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/troubleshooting
* https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-regions-limits-virtual-network
* https://learn.microsoft.com/en-us/azure/foundry/concepts/rbac-foundry
* https://pypi.org/project/azure-ai-projects/
* https://pypi.org/project/azure-ai-evaluation/
* https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/samples/evaluations/README.md

### Prerequisites

Cloud evaluation writes results to the project's connected **Azure Storage Account** via the project's **Managed Identity (MI)**.

#### 1. RBAC roles

| Identity | Role | Scope | Purpose |
|----------|------|-------|---------|
| **Project MI** | **Storage Blob Data Contributor** | Eval storage account | Read/write eval data and results ([troubleshooting](https://learn.microsoft.com/azure/foundry/observability/how-to/troubleshooting#missing-rbac-role-assignment-for-entra-id-authentication)) |
| **Project MI** | **Azure AI User** | Foundry **account** (parent) | Access the RAI safety service for safety/AI-assisted evaluators ([docs](https://learn.microsoft.com/azure/foundry/concepts/evaluation-regions-limits-virtual-network#virtual-network-support-for-evaluation)) |
| **User principal** | **Azure AI User** | Foundry **project** | Call the evaluation API from your notebook |

#### 2. Storage account network access

When the storage connection uses **Entra ID** (`authType: AAD`), the eval storage account **must** have `publicNetworkAccess: Enabled`. Otherwise the eval service gets a `403 AuthorizationFailure` even with correct RBAC roles.

See: [troubleshooting > Storage account network access restrictions](https://learn.microsoft.com/azure/foundry/observability/how-to/troubleshooting#storage-account-network-access-restrictions)

> **Tip**: The diagnostic cell below automatically checks the storage account network setting and enables public access if needed.

### Prepare evaluation data

Extract query/response from the agent run and save as JSONL for cloud evaluation.

Reference:
* https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/cloud-evaluation#prepare-input-data

In [48]:
# Build evaluation input from the v2 response object
eval_input = {
    "query": MESSAGE,
    "response": response.output_text,
    "ground_truth": "yes, I mailed you the weather info for Seattle",
}

# Ensure data directory exists
dir_path = os.path.join(os.getcwd(), "data")
os.makedirs(dir_path, exist_ok=True)

# Save as JSONL (one JSON object per line)
eval_file_path = os.path.join(dir_path, "singleagentrun.jsonl")
with open(eval_file_path, "w") as f:
    f.write(json.dumps(eval_input))

print(f"Saved eval data to {eval_file_path}")
print(json.dumps(eval_input, indent=2))

Saved eval data to /Users/yingding/Code/VCS/agents/ai-foundry-workshop/01-foundamentals-v3/data/singleagentrun.jsonl
{
  "query": "Can you email me weather info for Seattle ? My email is user@microsoft.com.",
  "response": "I have sent you an email with the current weather information for Seattle. It is rainy and 14\u00b0C there. If you need any more information, feel free to ask!",
  "ground_truth": "yes, I mailed you the weather info for Seattle"
}


### Run cloud evaluation (inline data)

Uses `SourceFileContent` to pass eval data directly in the request — no blob storage upload needed.

In [49]:
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

# Prepare inline data source — no blob storage upload required
inline_source = SourceFileContent(
    type="file_content",
    content=[
        SourceFileContentContent(item=eval_input),
    ],
)
print(f"Inline eval data prepared with {len(inline_source['content'])} item(s)")

Inline eval data prepared with 1 item(s)


In [50]:
from utils.blocklist_helper import resolve_account_location, _mask_sub_id
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.resourcegraph import ResourceGraphClient
from azure.mgmt.resourcegraph.models import QueryRequest

sub_id, rg, account_name = resolve_account_location(
    credential=credential,
    project_endpoint=settings.project_endpoint,
)
print(f"resolved > sub={_mask_sub_id(sub_id, True)} rg={rg} account={account_name}")

# Get project MI principal ID
cs_mgmt = CognitiveServicesManagementClient(credential, sub_id)
acct = cs_mgmt.accounts.get(rg, account_name)
mi_principal_id = acct.identity.principal_id if acct.identity else None
print(f"Project MI principal ID: {_mask_sub_id(mi_principal_id, True)}" if mi_principal_id else "No MI found")

# Find ALL storage accounts in the same resource group via ARG
rg_client = ResourceGraphClient(credential)
query = (
    f"Resources "
    f"| where type =~ 'microsoft.storage/storageaccounts' "
    f"   and resourceGroup =~ '{rg}' "
    f"| project name, id"
)
result = rg_client.resources(QueryRequest(query=query, subscriptions=[sub_id]))
storage_accounts = [(r["name"], r["id"]) for r in (result.data or [])]

print(f"\nStorage accounts in {rg}:")
for sa_name, sa_id in storage_accounts:
    print(f"  - {sa_name}")

# Generate az CLI commands to grant Storage Blob Data Owner to the project MI
# if mi_principal_id and storage_accounts:
#     for sa_name, sa_id in storage_accounts:
#         print(f"\n--- Grant MI access to {sa_name} ---")
#         print(f'az role assignment create \\')
#         print(f'  --assignee-object-id {mi_principal_id} \\')
#         print(f'  --assignee-principal-type ServicePrincipal \\')
#         print(f'  --role "Storage Blob Data Owner" \\')
#         print(f'  --scope "{sa_id}"')

resolved > sub=6753****-****-.... rg=rg-ai-sandbox-yw-uno account=foundry-proj-yw-uno-resource
Project MI principal ID: 06a2****-****-....

Storage accounts in rg-ai-sandbox-yw-uno:
  - aihubdemoywstoreacctdos
  - aihubdemoywuno
  - dshubdemoywuno1601061371
  - logicappmcptool909789
  - logicappmcptool949124
  - logicappmcptool984976
  - logicappmcpywun031338
  - stacctaievalywuno


In [51]:
import uuid
from urllib.parse import urlparse
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters
from azure.mgmt.storage import StorageManagementClient
import requests as _req

# --- 1. Resolve the PROJECT-level MI ---
project_name = urlparse(settings.project_endpoint).path.rstrip("/").rsplit("/", 1)[-1]

arm_token = credential.get_token("https://management.azure.com/.default").token
project_url = (
    f"https://management.azure.com/subscriptions/{sub_id}"
    f"/resourceGroups/{rg}"
    f"/providers/Microsoft.CognitiveServices/accounts/{account_name}"
    f"/projects/{project_name}?api-version=2025-04-01-preview"
)
resp = _req.get(project_url, headers={"Authorization": f"Bearer {arm_token}"})
resp.raise_for_status()
project_mi_id = resp.json().get("identity", {}).get("principalId")
print(f"Project MI: {_mask_sub_id(project_mi_id, True)}")

# --- 2. Find the connected eval storage account ---
connections_url = (
    f"https://management.azure.com/subscriptions/{sub_id}"
    f"/resourceGroups/{rg}"
    f"/providers/Microsoft.CognitiveServices/accounts/{account_name}"
    f"/projects/{project_name}/connections?api-version=2025-04-01-preview"
)
conn_resp = _req.get(connections_url, headers={"Authorization": f"Bearer {arm_token}"})
conn_resp.raise_for_status()

eval_sa_name = None
eval_sa_auth = None
for c in conn_resp.json().get("value", []):
    props = c.get("properties", {})
    if props.get("category") == "AzureStorageAccount" and props.get("isDefault", False):
        target = props.get("target", "")
        eval_sa_name = target.split("//")[-1].split(".")[0]
        eval_sa_auth = props.get("authType", "unknown")
        break

if not eval_sa_name:
    print("⚠️  No default AzureStorageAccount connection found on this project")
else:
    print(f"Eval storage: {eval_sa_name} (auth: {eval_sa_auth})")

# --- 3. Check storage account public network access ---
if eval_sa_name and eval_sa_auth == "AAD":
    storage_mgmt = StorageManagementClient(credential, sub_id)
    sa = storage_mgmt.storage_accounts.get_properties(rg, eval_sa_name)
    public_access = sa.public_network_access

    if public_access and public_access.lower() == "enabled":
        print(f"✅ publicNetworkAccess: Enabled")
    else:
        print(f"⚠️  publicNetworkAccess: {public_access} — enabling it for eval...")
        sa.public_network_access = "Enabled"
        storage_mgmt.storage_accounts.update(rg, eval_sa_name, {"public_network_access": "Enabled"})
        print(f"✅ publicNetworkAccess: Enabled (updated)")
elif eval_sa_name:
    print(f"ℹ️  Storage auth is '{eval_sa_auth}' (not AAD) — network check not needed")

# --- 4. Verify RBAC: project MI needs Storage Blob Data Contributor on eval storage ---
if eval_sa_name and project_mi_id:
    auth_client = AuthorizationManagementClient(credential, sub_id)
    sa_scope = (
        f"/subscriptions/{sub_id}/resourceGroups/{rg}"
        f"/providers/Microsoft.Storage/storageAccounts/{eval_sa_name}"
    )
    STORAGE_BLOB_DATA_CONTRIBUTOR = "ba92f5b4-2d11-453d-a403-e96b0029c9fe"
    existing = [
        a for a in auth_client.role_assignments.list_for_scope(
            sa_scope, filter=f"principalId eq '{project_mi_id}'"
        )
        if a.role_definition_id.endswith(STORAGE_BLOB_DATA_CONTRIBUTOR)
    ]
    if existing:
        print(f"✅ Project MI has Storage Blob Data Contributor on {eval_sa_name}")
    else:
        role_def_id = f"/subscriptions/{sub_id}/providers/Microsoft.Authorization/roleDefinitions/{STORAGE_BLOB_DATA_CONTRIBUTOR}"
        auth_client.role_assignments.create(
            scope=sa_scope,
            role_assignment_name=str(uuid.uuid4()),
            parameters=RoleAssignmentCreateParameters(
                role_definition_id=role_def_id,
                principal_id=project_mi_id,
                principal_type="ServicePrincipal",
            ),
        )
        print(f"✅ Granted Storage Blob Data Contributor to project MI on {eval_sa_name}")

# --- 5. Verify RBAC: project MI needs Azure AI User on Foundry account ---
    account_scope = (
        f"/subscriptions/{sub_id}/resourceGroups/{rg}"
        f"/providers/Microsoft.CognitiveServices/accounts/{account_name}"
    )
    AZURE_AI_USER = "53ca6127-db72-4b80-b1b0-d745d6d5456d"
    existing_ai = [
        a for a in auth_client.role_assignments.list_for_scope(
            account_scope, filter=f"principalId eq '{project_mi_id}'"
        )
        if a.role_definition_id.endswith(AZURE_AI_USER)
    ]
    if existing_ai:
        print(f"✅ Project MI has Azure AI User on {account_name}")
    else:
        role_def_id = f"/subscriptions/{sub_id}/providers/Microsoft.Authorization/roleDefinitions/{AZURE_AI_USER}"
        auth_client.role_assignments.create(
            scope=account_scope,
            role_assignment_name=str(uuid.uuid4()),
            parameters=RoleAssignmentCreateParameters(
                role_definition_id=role_def_id,
                principal_id=project_mi_id,
                principal_type="ServicePrincipal",
            ),
        )
        print(f"✅ Granted Azure AI User to project MI on {account_name}")

Project MI: 4ef8****-****-....
Eval storage: stacctaievalywuno (auth: AAD)
✅ publicNetworkAccess: Enabled
✅ Project MI has Storage Blob Data Contributor on stacctaievalywuno
✅ Project MI has Azure AI User on foundry-proj-yw-uno-resource


#### Textual similarity evaluators

* https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/textual-similarity-evaluators
* https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/cloud-evaluation#dataset-evaluation
* https://learn.microsoft.com/en-us/azure/foundry/concepts/observability#what-are-evaluators
* Config examples https://github.com/Azure-Samples/azureai-samples/blob/main/scenarios/evaluate/Supported_Evaluation_Metrics/AI_Judge_Evaluators_Quality/AI_Judge_Evaluators_Quality.ipynb
* SDK samples https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/samples/evaluations/sample_evaluations_builtin_with_dataset_id.py

In [55]:
# --- 1. Upload eval data via OpenAI files API (uses Foundry-managed storage) ---
# Check if a file with the same name already exists
EVAL_FILE_NAME = os.path.basename(eval_file_path)  # "singleagentrun.jsonl"
existing_files = [
    f for f in openai_client.files.list()
    if f.filename == EVAL_FILE_NAME and f.purpose == "evals"
]

if existing_files:
    # Reuse the most recent upload
    data_file = existing_files[-1]
    print(f"Reusing existing file: {data_file.id} ({data_file.filename})")
else:
    data_file = openai_client.files.create(
        file=open(eval_file_path, "rb"),
        purpose="evals",
    )
    print(f"Uploaded new file: {data_file.id} ({data_file.filename})")

OVERRIDE_UPLOAD = False  # Set True to force re-upload
if OVERRIDE_UPLOAD and existing_files:
    data_file = openai_client.files.create(
        file=open(eval_file_path, "rb"),
        purpose="evals",
    )
    print(f"Override: uploaded new version: {data_file.id}")

data_id = data_file.id

# --- 2. Define the data schema matching the JSONL fields ---
data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
            "ground_truth": {"type": "string"},
        },
        "required": [],
    },
    include_sample_schema=True,
)

# --- 3. Define testing criteria (evaluators) ---
# Ref: https://learn.microsoft.com/azure/foundry/concepts/observability#what-are-evaluators
testing_criteria = [
    # AI-assisted quality evaluator (requires deployment_name)
    {
        "type": "azure_ai_evaluator",
        "name": "relevance",
        "evaluator_name": "builtin.relevance",
        "initialization_parameters": {
            "deployment_name": settings.model_deployment_name,
        },
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    },
    # Safety evaluator (uses hosted RAI service)
    {
        "type": "azure_ai_evaluator",
        "name": "violence",
        "evaluator_name": "builtin.violence",
        "initialization_parameters": {
            "deployment_name": settings.model_deployment_name,
        },
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    },
    # NLP text-similarity evaluators (no model needed)
    {
        "type": "azure_ai_evaluator",
        "name": "bleu_score",
        "evaluator_name": "builtin.bleu_score",
        "data_mapping": {
            "response": "{{item.response}}",
            "ground_truth": "{{item.ground_truth}}",
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "f1_score",
        "evaluator_name": "builtin.f1_score",
        "data_mapping": {
            "response": "{{item.response}}",
            "ground_truth": "{{item.ground_truth}}",
        },
    },
    {
        "type": "azure_ai_evaluator",
        "name": "meteor_score",
        "evaluator_name": "builtin.meteor_score",
        "data_mapping": {
            "response": "{{item.response}}",
            "ground_truth": "{{item.ground_truth}}",
        },
    },
]

# --- 4. Create the evaluation ---
eval_object = openai_client.evals.create(
    name=f"Single Agent Eval {timestamp}",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)
print(f"Evaluation created: {eval_object.id}")

# --- 5. Create a run using uploaded file_id ---
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=f"dataset-run-{timestamp}",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileID(
            type="file_id",
            id=data_id,
        ),
    ),
)
print(f"Evaluation run created: {eval_run.id}")

Reusing existing file: file-ddb737c12ad141bfa78bf11ed834fc27 (singleagentrun.jsonl)
Evaluation created: eval_66950663a7e841628d457e90ad54e4f2
Evaluation run created: evalrun_dc59688b7ad04891a5dc107939449e27


In [56]:
import time

# Poll for evaluation run completion
while True:
    run_status = openai_client.evals.runs.retrieve(
        run_id=eval_run.id, eval_id=eval_object.id
    )
    if run_status.status in ("completed", "failed"):
        break
    print(f"Status: {run_status.status} ...")
    time.sleep(5)

print(f"\nEvaluation run finished: {run_status.status}")
if run_status.status == "completed":
    print(f"Result counts: {run_status.result_counts}")
elif run_status.status == "failed":
    print(f"Error: {run_status.error}")


Evaluation run finished: completed
Result counts: ResultCounts(errored=0, failed=1, passed=0, total=1)


### Foundry Project Evaluation Metric Dashboard

1. Login to `https://ai.azure.com/`
2. choose your Foundry Project
3. Open the `Evaluation` menu item
4. choose an evaluation run to see metric dashboard

![](imgs/cloud_eval_metric_dashboard.png)